# 01 - Phase 1: Reproducing the STTran Baseline

**One-liner:** rebuild the original STTran pipeline (Faster R-CNN + ResNet-101 backbone -> STTran graph head) on Action Genome, train it, confirm the published numbers (~24.6 mAP, ~71.8 PredCLS R@20).

If we can't reproduce the baseline, the comparisons in Phase 2/3 are meaningless. So Phase 1 is the foundation.

## What was the goal of Phase 1?

From `knowledge information/st_project_proposal.pdf` and `knowledge information/Project Progress.pdf`:

> Build the original STTran (Spatial-Temporal Transformer) system using a ResNet-101 architecture. The target baseline performance for the system requires 71.8% Recall@20 results on PredCLS.

Translation: hit the headline numbers from Cong et al., ICCV 2021 on Action Genome, on our SJSU H100 hardware, with our copy of the codebase. Don't innovate yet.

## The architecture in Phase 1

```mermaid
flowchart LR
    Frame[AG frame] --> ResNet[ResNet-101 backbone]
    ResNet --> RCNNBase[fasterRCNN.RCNN_base]
    RCNNBase --> RPN[RPN]
    RPN --> RoI[RoI Align 7x7]
    RoI --> ClsBox["RCNN_cls_score + RCNN_bbox_pred"]
    ClsBox --> STTran[STTran heads]
    STTran --> Out["attention / spatial / contacting predictions"]
```

Standard "detector first" flow. The whole stack is in `STTran/lib/object_detector.py` - take a careful look at the constructor at lines 111-224 and the GT-box forward path at lines 719-859.

## Where ResNet-101 enters the code

Inside the `detector` class constructor in `STTran/lib/object_detector.py`:

```python
self.fasterRCNN = resnet(classes=self.object_classes, num_layers=101, pretrained=False, class_agnostic=False)
self.fasterRCNN.create_architecture()
```
(line 157)

And later, when forwarding an image, `_extract_base_features` decides which backbone to use:

```python
def _extract_base_features(self, images):
    if self.backbone_name == 'vitdet':
        feat_dict = self.vitdet(images)
        return feat_dict['base']
    return self.fasterRCNN.RCNN_base(images)
```
(line 234)

So in Phase 1 (`backbone_name = 'resnet101'`) every base-feature extraction goes through the legacy ResNet path. **This is exactly the path that the Mixed-Path Bug used to leak into Phase 2** - we'll see that in notebook 02.

## Pretrained weights: where do they come from?

The original Cong et al. release ships with a Faster R-CNN checkpoint already fine-tuned on Action Genome (`fasterRCNN/models/faster_rcnn_ag.pth`). Phase 1 simply loads it:

```python
if load_ag_ckpt:
    checkpoint = torch.load('fasterRCNN/models/faster_rcnn_ag.pth', map_location='cpu')
    detector_state = checkpoint['model'] if isinstance(checkpoint, dict) and 'model' in checkpoint else checkpoint
    ...
    missing_keys, unexpected_keys = self.fasterRCNN.load_state_dict(detector_state, strict=False)
```
(`STTran/lib/object_detector.py` lines 159-181)

**Why this matters:** because the AG checkpoint already exists, Phase 1 doesn't actually need to retrain the detector from scratch - it just loads the published weights and trains the STTran graph head on top. Phase 2/3 are the ones that re-do detector pretraining, because the new ViT/DINOv2 backbones produce features that `faster_rcnn_ag.pth` was never trained on.

**Asked-about gotcha:** Phase 1's RCNN heads are *not* reinitialized (`DETECTOR_REINIT_ROI_HEADS=0`). Phase 2/3 do reinitialize them (`DETECTOR_REINIT_ROI_HEADS=1`) because the feature distribution coming out of a ViT is unrelated to what the original Faster R-CNN cls/bbox heads expect.

## Stage 1 vs Stage 2 in Phase 1

- **Stage 1**: detector training. Only object classification + bbox regression + RPN losses. In Phase 1 this is essentially a sanity check / fine-tune of the supplied AG checkpoint. Code: `STTran/train_detector_stage1.py`.
- **Stage 2**: freeze the detector, train the STTran graph head with relation losses. Code: `STTran/train.py`. The relation losses are computed for the three families:
  - attention: CE
  - spatial: MLM (multi-label margin) or BCE (configurable via `--bce_loss`)
  - contacting: MLM or BCE

Loss aggregation in `STTran/train.py` line 1045-1053:
```python
relation_loss = (
    losses['attention_relation_loss']
    + losses['spatial_relation_loss']
    + losses['contact_relation_loss']
)
if 'object_loss' in losses:
    loss = stage2_obj_weight * losses['object_loss'] + stage2_rel_weight * relation_loss
else:
    loss = stage2_rel_weight * relation_loss
```

**Mental model:** Phase 1's whole job is to *validate the harness*. We are checking that data loading, evaluation harness, training loop, and the AG checkpoint all behave as they should before we touch anything.

## Dataset and runtime filtering

The Action Genome split numbers in your final report are *post-filter*:

| Split | Raw videos | Filtered videos | Filtered frames |
|---|---|---|---|
| Train | 7,649 | 7,649 - (175 peopleless) - (87 single-frame) = ~7,387 effective | **167,068** |
| Eval (SGCLS/SGDET)  | 1,737 | 1,737 - 48 - 20 = ~1,669 effective | **54,429** |
| Eval (PredCLS) | 1,750 | slightly less filtering | 56,923 |

Filters applied at runtime by the dataloaders (`STTran/dataloader/action_genome.py`, `action_genome_detector.py`):

1. **No `person_bbox` in the frame** -> drop the frame.
2. **Single-frame video** -> drop the video (STTran needs a temporal window).
3. **Empty annotation** -> drop the frame.

**Asked-about gotcha:** the PredCLS split has slightly fewer filters applied (1750 videos / 56923 frames vs 1737 / 54429 for SGCLS/SGDET). That difference is by design - PredCLS doesn't need to drop frames where the detector would have failed. From `Big Picture.pdf`: *"That split difference was diagnosed and is expected, not a bug."*

## Hardware and config for Phase 1

Single H100 GPU on the SJSU HPC cluster (`condo7`/`condo8`/`condo9`). Standard config (you can find it in `STTran/scripts/train_sttran_h100.sbatch`):

- `BACKBONE=resnet101`
- `DETECTOR_BN_MODE=batchnorm` (default for ResNet)
- input size: original STTran resize-and-pad recipe (around 600 short side)
- Stage 2 epochs: 10
- Stage 2 LR: 1e-5
- batch size: 1 video at a time (videos are variable-length)
- BCE loss for spatial/contact relations (`BCE_LOSS=1`)

## Phase 1 results

We achieved parity with the published STTran paper:

| Metric | Base STTran paper | Our Phase 1 | Match? |
|---|---|---|---|
| Detector mAP@0.5 | 24.6 | ~24.6 | yes |
| PredCLS R@20 (With) | 71.8 | ~71.8 | yes |
| SGCLS R@20 (With) | 47.5 | ~47.5 | yes |
| SGDET R@20 (With) | 34.1 | ~34.1 | yes |

**The numbers we report in our final paper for "Base STTran Paper" are the published numbers, not our Phase 1 reproduction**, because the published numbers are what the field cites. Our Phase 1 reproduction served as a sanity check - it confirmed our harness was correct so that any later deviation would be attributable to the backbone, not our infra.

## What Phase 1 unlocked

Three things, all of which mattered later:

1. **A working data path.** AG raw `.pkl` annotations + Charades raw video -> filtered `(167,068, 54,429)` train/eval frame sets, reproducible.
2. **A trustworthy evaluation harness** for PredCLS / SGCLS / SGDET across With/Semi/No constraints.
3. **A reference checkpoint** (`faster_rcnn_ag.pth`) and a reference scorecard so we know exactly what "correct" looks like before changing anything.

When Phase 2 SGCLS came back at 0.013 R@20 (yes, *thirteen-thousandths*), Phase 1's parity is what told us "the harness is fine - the bug is upstream in the new ViT path."

## How to talk about Phase 1 in 30 seconds

> "Phase 1 was reproducing the STTran baseline on Action Genome with the original ResNet-101 backbone. We trained Stage 1 detector pretraining and Stage 2 graph training on a single H100, hit the published numbers - 24.6 detector mAP, 71.8 PredCLS R@20, 47.5 SGCLS R@20, 34.1 SGDET R@20 - and that gave us a trustworthy baseline so the Phase 2/3 backbone swaps would be a clean comparison."

If asked "why did you bother reproducing it?" - because without parity, every later number is a debugging artifact, not a result.

---
**Next:** open `02_phase2_vit_bugs_and_fixes.ipynb` for the heart of the story - the three bugs we hunted down.